In [ ]:
import os
import sys

from matplotlib import pyplot as plt
import mlflow
import pandas as pd
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
import torch
import seaborn as sns

sys.path.append(os.path.abspath(os.path.join('..')))
from models import GNNModel
from torch_geometric.loader import DataLoader
from utils import mol_to_graph
from utils import MLFlowManager


from joblib import Parallel, delayed

def log_regression_plots(y_true, y_pred, run_name):
    plt.figure(figsize=(8, 6))
    sns.scatterplot(x=y_true, y=y_pred, alpha=0.5)
    plt.plot([min(y_true), max(y_true)], [min(y_true), max(y_true)], '--r', lw=2)
    plt.xlabel("Actual pIC50")
    plt.ylabel("Predicted pIC50")
    plt.title(f"Regression Fit - {run_name}")
    
    plot_path = "pred_vs_actual.png"
    plt.savefig(plot_path)
    mlflow.log_artifact(plot_path)
    plt.close()

parquet_path = "parquets/df_ml_with_scaffold.parquet"
df = pd.read_parquet(parquet_path)

print("SMILES to graph conversion...")
dataset = Parallel(n_jobs=-1)(
    delayed(mol_to_graph)(s, y) for s, y in zip(df['canonical_smiles'], df['pic50'])
)
# dataset = [mol_to_graph(s, y) for s, y in zip(df['canonical_smiles'], df['pic50'])]
dataset = [d for d in dataset if d is not None] 

mf = MLFlowManager(experiment_name="ChEMBL_GNN_Scaffold_Split")

model_types = ["GCN", "GIN"]
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
train_data, test_data = train_test_split(dataset, test_size=0.2, random_state=42)
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)
for m_type in model_types:
    model = GNNModel(num_node_features=4, hidden_channels=64, model_type=m_type).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
        
    run_name = f"Run_{m_type}_ScaffoldData"
    print(f"Starting: {run_name}")
    
    # 1. Trening - upewnij się, że ta metoda zwraca run_id lub zostawia otwarty run
    model.train_gnn(
        model=model, 
        loader=train_loader, 
        optimizer=optimizer, 
        device=device, 
        mf_manager=mf, 
        run_name=run_name
    )

    # 2. Ewaluacja
    model.eval()
    y_true, y_pred = [], []
    
    with torch.no_grad():
        for data in test_loader:
            data = data.to(device)
            # Spłaszczamy output do 1D, aby zgadzał się z y
            out = model(data.x, data.edge_index, data.batch).view(-1)
            y_true.extend(data.y.cpu().numpy())
            y_pred.extend(out.cpu().numpy())
    
    # 3. Obliczenia
    final_r2 = r2_score(y_true, y_pred)
    final_mae = mean_absolute_error(y_true, y_pred)

    # 4. Logowanie do aktywnego runu MLflow
    # Używamy start_run z existing run_name, aby dopisać metryki i wykresy do tego samego miejsca
    with mlflow.start_run(run_name=run_name, nested=True):
        mlflow.log_metric("final_test_r2", final_r2)
        mlflow.log_metric("final_test_mae", final_mae)
        
        # Wywołanie Twojej funkcji wykresu
        log_regression_plots(y_true, y_pred, m_type)

    print(f"Finished {m_type}: R2={final_r2:.4f}, MAE={final_mae:.4f}")


SMILES to graph conversion...


[17:59:20] Explicit valence for atom # 17 P, 7, is greater than permitted
[18:04:14] Explicit valence for atom # 19 P, 7, is greater than permitted
[18:06:54] Explicit valence for atom # 1 P, 7, is greater than permitted
[18:07:24] Explicit valence for atom # 1 P, 7, is greater than permitted
[18:09:26] Explicit valence for atom # 16 P, 7, is greater than permitted
[18:09:51] Explicit valence for atom # 1 P, 7, is greater than permitted
2026/05/09 18:09:56 INFO mlflow.tracking.fluent: Experiment with name 'ChEMBL_GNN_Scaffold_Split' does not exist. Creating a new experiment.


Starting: Run_GCN_ScaffoldData


2026/05/09 18:14:28 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/09 18:14:28 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.


Finished Run_GCN_ScaffoldData with Loss: 2.2516
🏃 View run Run_GCN_ScaffoldData at: http://localhost:5000/#/experiments/4/runs/a86c4c87975e41e299f4b7c90e4205e1
🧪 View experiment at: http://localhost:5000/#/experiments/4
Starting: Run_GIN_ScaffoldData


2026/05/09 18:18:34 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/09 18:18:34 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.


Finished Run_GIN_ScaffoldData with Loss: 2.0586
🏃 View run Run_GIN_ScaffoldData at: http://localhost:5000/#/experiments/4/runs/3611356b230c47ecbed04dec26ac5bb7
🧪 View experiment at: http://localhost:5000/#/experiments/4
